# Tutorial 1: Quick Start

This notebook mirrors the quick-start flow from the getting started guide.



## What is `floodsr`?

`floodsr` is a flood-depth resolution enhancement tool.
It takes a low-resolution depth raster and reconstructs a higher-resolution result using terrain context from a DEM.

## Using this tutorial

There are three common ways to run this tutorial (i.e., *execution contexts*):
- **command line interface (CLI)**: proceed by copy/pasting the below commands from your web-browser into your terminal. NOTE for most terminals you'll need to remove the prefix `!` from each command.
- **local notebook (Jupyter)**: Use the <i class="fas fa-download"></i> button to save this notebook as an `.ipynb` file to your local machine (may need to click *save as*) and open it with your local Jupyter kernel. see {ref}`basic_install_local_notebook` for details.
- **hosted notebook (Colab)**: Use the <i class="fas fa-rocket"></i> button to open this tutorial on Google Colab.


## Install `floodsr`

Don't be scared. Install is super easy.

The appropriate installation steps depend on your execution context, which is described in detail in {ref}`basic_install`.
Here are the basic steps for each:
- **command line interface (CLI)**: first, ensure pipx is installed and on PATH, then run `pipx install floodsr`. see {ref}`basic_install_cli` for more details.
- **local notebook (Jupyter)**: same as above. then either follow the additional steps in {ref}`basic_install_local_notebook` to properly set up your environment, or be lazy and uncomment the below Jupyter cell and run.
- **hosted notebook (Colab)**: uncomment the below colab cell and run. see {ref}`basic_install_google_colab` for details.

command line interface (CLI) install: run `pipx install floodsr` in your terminal

Local notebook (Jupyter) magic install

In [1]:

# %pip install -q floodsr

Hosted notebook (Colab) install

In [2]:
# !python -m pip install -q floodsr

## prove the install

run the below command to confirm the install worked and `floodsr` is on your PATH:

In [3]:
# HIDDEN (tags:["remove-input"])
# cell below has tag:'remove-output'


In [4]:
!floodsr --help

usage: floodsr [-h] [--version] [-v] [-q]
               [--log-level {DEBUG,INFO,WARNING,ERROR}]
               {models,tohr,doctor} ...

Run FloodSR model, cache, and runtime utility commands.

positional arguments:
  {models,tohr,doctor}
    models              List manifest models or fetch cached model weights.
    tohr                Run one super-resolution pass for a low-res depth
                        raster.
    doctor              Report runtime dependency and provider diagnostics.

options:
  -h, --help            show this help message and exit
  --version             Print the installed FloodSR package version and exit.
  -v, --verbose         Increase logging verbosity (repeatable).
  -q, --quiet           Decrease logging verbosity (repeatable).
  --log-level {DEBUG,INFO,WARNING,ERROR}
                        Explicit log level override.


## Download test data

Before running commands, download a small example dataset into your current working directory.

In [5]:
from urllib.request import urlretrieve

urlretrieve(
    "https://github.com/cefect/floodsr/releases/download/v0.0.3/hires002_dem.tif",
    "hires002_dem.tif",
)
urlretrieve(
    "https://github.com/cefect/floodsr/releases/download/v0.0.3/lowres032.tif",
    "lowres032.tif",
)


('lowres032.tif', <http.client.HTTPMessage at 0x783c241c0950>)

## List available models

Before running inference, inspect the manifest to see which model versions are available.

In [6]:
!floodsr models list

CostGrow_Terrain	(built-in, no download)	Rules-based terrain-penalty CostGrow method (fluvial only)
ResUNet_16x_DEM	model_infer.onnx	https://github.com/cefect/floodsr/releases/download/v2026.02.19/model_infer.onnx


## Fetch model weights

Fetch the default model into the local cache.

In [7]:
!floodsr models fetch --no-progress ResUNet_16x_DEM


INFO:floodsr.model_registry:attempting unauthenticated model download from
    https://github.com/cefect/floodsr/releases/download/v2026.02.19/model_infer.onnx


ERROR:floodsr.cli:file does not exist: /tmp/.cache/floodsr/ResUNet_16x_DEM/model_infer.onnx.part


## Run `tohr` with a local DEM

Now let's run the **to high resolution** `tohr` command to enhance the resolution of our low-resolution depth raster against the hi-res DEM.

In [8]:
# HIDDEN (tags:["remove-input"])
# cell below has tag:'remove-output'


In [9]:
!floodsr tohr --in lowres032.tif --dem hires002_dem.tif


INFO:floodsr.cli:loaded ORT model 'model_infer.onnx' with providers=['CPUExecutionProvider'] and scale=16
INFO:floodsr.cli:tohr path selection
  requested_window_method=feather
  dem_float32_bytes=1,048,576
  windowed_io_threshold_bytes=33,554,432
  selected_platform_materialization=simple


INFO:floodsr.cli:starting tohr inference with model_version=ResUNet_16x_DEM
model
    /tmp/.cache/floodsr/ResUNet_16x_DEM/model_infer.onnx
platform_depth_lr
    /home/cefect/LS/10_IO/2407_FHIMP/tmp/floodsr-tutorial_1-IYVRbq/tmp/floodsr-platform-prep-qzkjmx9x/lowres032_platform_depth.tif
platform_dem_hr
    /home/cefect/LS/10_IO/2407_FHIMP/tmp/floodsr-tutorial_1-IYVRbq/tmp/floodsr-platform-prep-qzkjmx9x/hires002_dem_platform_dem.tif
output
    /home/cefect/LS/10_IO/2407_FHIMP/tmp/floodsr-tutorial_1-IYVRbq/run/lowres032_sr.tif
INFO:floodsr.cli:platform-preprocessed inputs
  depth_lr shape=(32, 32) res=(32.0, 32.0) m/pix
  dem_hr shape=(512, 512) res=(2.0, 2.0) m/pix
INFO:floodsr.cli:model preprocessing complete
INFO:floodsr.cli:tohr execution path
  window_method=feather
  execution_path=simple
  model_space_shape=(512, 512)
  raw_output_shape=(512, 512)
INFO:floodsr.cli:prepared inputs summary:
  aligned depth_lr shape=(32, 32) res=(32.0, 32.0) m/pix
  aligned dem_hr shape=(512, 512) re

feather pass: 100%|███████████████████████████| 1/1 [00:00<00:00,  7.64window/s]

final write pass: 100%|██████████████████| 128/128 [00:00<00:00, 7048.51block/s]
INFO:floodsr.cli:finished tohr inference in 0.238s; wrote 1,116,817 bytes to
    /home/cefect/LS/10_IO/2407_FHIMP/tmp/floodsr-tutorial_1-IYVRbq/run/lowres032_sr.tif
/home/cefect/LS/10_IO/2407_FHIMP/tmp/floodsr-tutorial_1-IYVRbq/run/lowres032_sr.tif


In [10]:
# HIDDEN (tags:["remove-input"])
# Check that the expected output raster exists.
assert __import__("pathlib").Path("lowres032_sr.tif").is_file(), "missing output raster\n    lowres032_sr.tif"

If you inspect your working directory, you should see a new file called `lowres032_sr.tif`. 
This is the output of the `tohr` command, and should match the resolution of the DEM (2m in this case).
Because we didn't specify an output path, `floodsr` defaulted to writing the output to the same directory as the input, with `_sr` appended to the filename.

Hooray! You just ran your first `floodsr` command. You deserve some <span style="font-size: 1.4em;">🍰</span>.

## Run `tohr` with fetched HRDEM data

Passing `--fetch-hrdem` tells `tohr` to attempt to fetch the HRDEM data for your raster extents. 
Of course, this will only work if your raster extents overlap with the coverage of the HRDEM data.
Otherwise, you'll get an error. 

In [11]:
# HIDDEN (tags:["remove-input"])
# cell below has tag:'remove-output'


In [12]:
!floodsr tohr --in lowres032.tif --fetch-hrdem


INFO:floodsr.cli:starting DEM fetch
  source_id=hrdem
  stac_url=https://datacube.services.geo.ca/api
  collection=hrdem-mosaic-1m
  asset_key=dtm
  gdal_available=True
  force_tiling=False
  fetch_window_size=8192
  memory_limit_gib=16.00
  stac_query_limit=200
  use_project_extent_filter=True
  use_cache=True
  cache_dir=None
  show_progress=True
  project_extent_url=https://maps-cartes.services.geo.ca/server_serveur/rest/services/NRCan/coverage_HRDEM_en/MapServer/4
  depth_lr_fp=
    /home/cefect/LS/10_IO/2407_FHIMP/tmp/floodsr-tutorial_1-IYVRbq/run/lowres032.tif


INFO:floodsr.cli:found 1 HRDEM item(s) intersecting low-res tile bounds after exact intersection filter
INFO:floodsr.cli:raw fetch request grid: width=1,024, height=1,024, pixels=1,048,576, non_windowed_peak_estimate=0.01 GiB
INFO:floodsr.cli:HRDEM tile cache state: enabled=1, dir=/tmp/.cache/floodsr/floodsr_hrdem_tile_cache, existing_files=0, request_token=11afd1814d4c09e5c44a8f97


INFO:floodsr.cli:wrote fetched HRDEM tile to
    /home/cefect/LS/10_IO/2407_FHIMP/tmp/floodsr-tutorial_1-IYVRbq/tmp/floodsr_hrdem_output_11afd1814d4c09e5c44a8f97.tif


INFO:floodsr.cli:loaded ORT model 'model_infer.onnx' with providers=['CPUExecutionProvider'] and scale=16
INFO:floodsr.cli:tohr path selection
  requested_window_method=feather
  dem_float32_bytes=4,194,304
  windowed_io_threshold_bytes=33,554,432
  selected_platform_materialization=simple


INFO:floodsr.cli:starting tohr inference with model_version=ResUNet_16x_DEM
model
    /tmp/.cache/floodsr/ResUNet_16x_DEM/model_infer.onnx
platform_depth_lr
    /home/cefect/LS/10_IO/2407_FHIMP/tmp/floodsr-tutorial_1-IYVRbq/tmp/floodsr-platform-prep-uhjw5zzk/lowres032_platform_depth.tif
platform_dem_hr
    /home/cefect/LS/10_IO/2407_FHIMP/tmp/floodsr-tutorial_1-IYVRbq/tmp/floodsr-platform-prep-uhjw5zzk/floodsr_hrdem_output_11afd1814d4c09e5c44a8f97_platform_dem.tif
output
    /home/cefect/LS/10_IO/2407_FHIMP/tmp/floodsr-tutorial_1-IYVRbq/run/lowres032_sr.tif


INFO:floodsr.cli:platform-preprocessed inputs
  depth_lr shape=(32, 32) res=(32.0, 32.0) m/pix
  dem_hr shape=(1024, 1024) res=(1.0, 1.0) m/pix
INFO:floodsr.cli:model preprocessing complete
INFO:floodsr.cli:tohr execution path
  window_method=feather
  execution_path=simple
  model_space_shape=(512, 512)
  raw_output_shape=(1024, 1024)
INFO:floodsr.cli:prepared inputs summary:
  aligned depth_lr shape=(32, 32) res=(32.0, 32.0) m/pix
  aligned dem_hr shape=(512, 512) res=(2.0, 2.0) m/pix
  max_depth=5.0
  dem_pct_clip=95.0
INFO:floodsr.cli:window config
  method=feather
  overlap_lr=8
  overlap_hr=128
  tile_size_lr=32
  tile_size_hr=512
INFO:floodsr.cli:running feather tiling over 1x1 grid
  stride_hr=384
  overlap_hr=128
feather pass:   0%|                                   | 0/1 [00:00<?, ?window/s]

feather pass: 100%|███████████████████████████| 1/1 [00:00<00:00, 19.20window/s]
INFO:floodsr.cli:post-resampling model output from (512, 512) to (1024, 1024) on raw DEM grid with bilinear interpolation.
post-resample pass:   0%|                               | 0/1 [00:00<?, ?step/s]

final write pass:   0%|                               | 0/16 [00:00<?, ?block/s]

final write pass: 100%|█████████████████████| 16/16 [00:00<00:00, 374.42block/s]
INFO:floodsr.cli:finished tohr inference in 0.184s; wrote 4,388,497 bytes to
    /home/cefect/LS/10_IO/2407_FHIMP/tmp/floodsr-tutorial_1-IYVRbq/run/lowres032_sr.tif


/home/cefect/LS/10_IO/2407_FHIMP/tmp/floodsr-tutorial_1-IYVRbq/run/lowres032_sr.tif


In [13]:
# HIDDEN (tags:["remove-input"])
# Check that the expected output raster exists.
assert __import__("pathlib").Path("lowres032_sr.tif").is_file(), "missing output raster\n    lowres032_sr.tif"

## Next steps

To help you on your hi-res flood journey, now is a good time to get familar with:
- the {doc}`../user_guide` for a more detailed walkthrough of the above steps and concepts.
- the {doc}`../cli_reference` for a detailed reference of all available commands and options.

Or continue with the next tutorial notebook